# TradeSense · Component 03 · Stage E · Notebook E2
### How well do simple order-book scores find the synthetic manipulation events? (the yardstick the TGNN has to beat)

**Why.** Before the TGNN is judged on the injected events (E2b, after D2), we need to know how much a *plain* score already achieves. If a trivial rule finds almost every event, the TGNN has little to add; if it finds only the easy ones, there is room. Both outcomes are reported honestly.

**Scores (no training except one table of numbers from training days).**
| id | score at interval *i* (uses only data up to *i*) |
|---|---|
| B1-top5 | per node, the size of the one-step change of the scaled log quantity divided by that node's average change on the **training** days; mean of the 5 largest nodes (primary) |
| B1-all | the same, mean over all 50 nodes (secondary) |
| B5 | robust z-score (median and MAD of the previous 60 s) of the total absolute quantity change over all nodes |

**Fair comparison rule.** Each score gets its own threshold so that it raises the **same number of false alarm episodes on clean time** (at least 30 s away from every injected event): 2, 5 and 10 episodes per symbol-day. An event is *found* if an alarm occurs between its start and one interval after its end. *Delay* = time from event start to the first alarm.

**What it uses.** The injected July sets from E1 (`TGNN_StageE_v1/val_injected`) and the two training days (only to measure the average per-node change). **No test day is opened. No GPU is needed** (CPU, about 10 to 25 minutes). Every number is a *synthetic injection* result.

## 1 · Settings (nothing to edit)

In [ ]:
import os, gc, sys, json, time, shutil, hashlib, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir(), f'Drive folder not found: {BASE}'
STAGEC_NAME = 'TGNN_StageC_GraphFeatures_v2'; E_NAME = 'TGNN_StageE_v1'
SYMS = ['BTCUSDT', 'ETHUSDT']; VAL_DAYS = ['2024-07-01']; TRAIN_DAYS = ['2024-05-01', '2024-06-01']
BURDENS = [2, 5, 10]       # clean false-alarm episodes per symbol-day
PRIMARY_BURDEN = 5; TOP = 5; ACT_WIN = 600; NEAR = 300; GAP = 100
print('burdens', BURDENS, '| primary', PRIMARY_BURDEN, '| top', TOP, '| activity window', ACT_WIN, '| clean distance', NEAR, '| episode gap', GAP)

Mounted at /content/drive
burdens [2, 5, 10] | primary 5 | top 5 | activity window 600 | clean distance 300 | episode gap 100


## 2 · Libraries (written to the Colab working folder, not to Drive)

In [ ]:
%%writefile stage_e_lib.py
"""TradeSense C03 - Stage E library: order-book manipulation INJECTOR (synthetic, labelled as synthetic).
Works on the frozen Stage C tensors (scaled float16). Levels are decoded to quantities, a pattern is added to the quantities of existing price levels
(prices, spread and mid never change), and ONLY the quantity-derived channels are recomputed with the same formulas as Stage C:
  node ch2 log_qty, ch3 d_log_qty, ch4 cum_log_depth, glob ch1 imbalance_top5   (ch1/ch5 distances and glob ch0/ch2 do not depend on quantities).
Intervals outside [start, end+1] of an event are copied untouched. Injection never targets a flagged interval. Training days are never injected."""
import json, os
from pathlib import Path
import numpy as np

K = 25; NN = 50; NCH = 6; GCH = 3; LOGQ, DLQ, CUM = 2, 3, 4; IMB = 1
SCALED_NODE = [1, 2, 3, 4, 5]
PATTERNS = {1: 'spoof', 2: 'layering', 3: 'quote_stuffing', 4: 'liquidity_vacuum'}
MAGS = {1: 'subtle', 2: 'medium', 3: 'easy'}
# magnitude parameters per pattern and level (1 subtle, 2 medium, 3 easy). multipliers are in units of the day-median quantity of that rank
PAR = {1: {1: 3.0, 2: 8.0, 3: 20.0}, 2: {1: 2.0, 2: 5.0, 3: 10.0}, 3: {1: 0.8, 2: 2.0, 3: 4.0}, 4: {1: 0.60, 2: 0.85, 3: 0.95}}
DUR = {1: (10, 100), 2: (20, 150), 3: (30, 100), 4: (10, 50)}          # in 100 ms intervals

# ---------------------------------------------------------------- decode / recompute
def _mu_sd(sc, ch): j = SCALED_NODE.index(ch); return float(sc['node_mean'][j]), float(sc['node_std'][j])
def decode_q(nodes, sc, a, b):
    """quantities of intervals [a,b) from the scaled log_qty channel (float64, >= 0)"""
    m, s = _mu_sd(sc, LOGQ); lq = nodes[a:b, :, LOGQ].astype(np.float64) * s + m; return np.maximum(np.expm1(lq), 0.0)
def encode_channels(q, q_prev, sc, glob_mean, glob_std):
    """scaled (log_qty, d_log_qty, cum_log_depth) of quantities q (m,50) given the quantities of the interval just before (50,) and the scaled imbalance (m,)"""
    lq = np.log1p(q); lq_prev = np.log1p(q_prev)[None]; dlq = lq - np.concatenate([lq_prev, lq[:-1]], 0)
    cum = np.concatenate([np.log1p(np.cumsum(q[:, :K], 1)), np.log1p(np.cumsum(q[:, K:], 1))], 1)
    s5b = q[:, :5].sum(1); s5a = q[:, K:K + 5].sum(1); imb = (s5b - s5a) / np.maximum(s5b + s5a, 1e-12)
    out = []
    for ch, arr in ((LOGQ, lq), (DLQ, dlq), (CUM, cum)): m, s = _mu_sd(sc, ch); out.append((arr - m) / s)
    return out[0], out[1], out[2], (imb - glob_mean) / glob_std

def day_ref(nodes, sc, flags, step=50):
    """per-rank median quantity of the day (sampled every `step` intervals, flag-free intervals only): the unit of injection magnitude"""
    idx = np.flatnonzero(flags[::step] == 0) * step
    m, s = _mu_sd(sc, LOGQ); lq = nodes[idx, :, LOGQ].astype(np.float64) * s + m; return np.median(np.maximum(np.expm1(lq), 0.0), 0)

# ---------------------------------------------------------------- pattern definitions: additive/multiplicative change of q over the event
def pattern_delta(q_span, pat, mag, side, rng, ref, dur):
    """returns the modified quantities q' for the intervals of the event (dur, 50)"""
    q = q_span.copy(); off = 0 if side == 0 else K; par = PAR[pat][mag]; meta = {}
    if pat == 1:                                                   # spoof: one large order a few ranks away from the touch, then cancelled
        r = int(rng.integers(1, 6)); q[:, off + r] += par * ref[off + r]; meta = {'ranks': [r]}
    elif pat == 2:                                                 # layering: 3-6 consecutive ranks, staggered placement, removed together
        L = int(rng.integers(3, 7)); r0 = int(rng.integers(1, 5)); ranks = list(range(r0, r0 + L)); meta = {'ranks': ranks}
        for i, r in enumerate(ranks):
            st = min(i * int(rng.integers(0, 5)), max(dur - 2, 0)); q[st:, off + r] += par * ref[off + r] * rng.uniform(0.7, 1.3)
    elif pat == 3:                                                 # quote stuffing: rapid on/off of quantity on the 10 best ranks
        on = (np.arange(dur) % 2 == 0); q[on, off:off + 10] *= (1.0 + par); meta = {'ranks': list(range(10))}
    elif pat == 4:                                                 # liquidity vacuum: the k best ranks lose most of their quantity
        k = int(rng.integers(3, 9)); q[:, off:off + k] *= (1.0 - par); meta = {'ranks': list(range(k))}
    return q, meta

# ---------------------------------------------------------------- placement
def place_events(flags, win_end, n_events, rng, margin=600, gap=600, durations=DUR, patterns=(1, 2, 3, 4), mags=(1, 2, 3)):
    """random event slots inside long flag-free stretches: [start, end] (inclusive), margin intervals of clean context before and 50 after, at least `gap` intervals between events.
    n_events = events per (pattern, magnitude) pair."""
    n = len(flags); bad = np.concatenate([[0], np.cumsum((flags != 0).astype(np.int64))]); taken = np.zeros(n, bool); ev = []
    jobs = [(p, m) for p in patterns for m in mags for _ in range(n_events)]; rng.shuffle(jobs)
    for p, m in jobs:
        dur = int(rng.integers(durations[p][0], durations[p][1] + 1))
        for _try in range(200):
            s = int(rng.integers(margin, n - dur - 60)); e = s + dur - 1
            if bad[e + 52] - bad[s - margin] != 0: continue                # whole stretch must be flag-free
            if taken[max(s - gap, 0):e + gap].any(): continue
            taken[s:e + 1] = True; ev.append({'start': s, 'end': e, 'pattern': p, 'mag': m, 'side': int(rng.integers(0, 2))}); break
    ev.sort(key=lambda d: d['start']); return ev

# ---------------------------------------------------------------- apply
def inject_day(nodes, glob, flags, sc, events, seed):
    """returns copies (nodes2, glob2) and an interval label array; events must come from place_events (flag-free, separated)"""
    nodes2 = nodes.copy(); glob2 = glob.copy(); n = len(flags); pat_arr = np.zeros(n, np.int8); mag_arr = np.zeros(n, np.int8); rng = np.random.default_rng(seed)
    ref = day_ref(nodes, sc, flags); gm = float(sc['glob_mean'][IMB]); gs = float(sc['glob_std'][IMB]); out = []
    for e in events:
        s, t, p, m, side = e['start'], e['end'], e['pattern'], e['mag'], e['side']; dur = t - s + 1
        q = decode_q(nodes, sc, s - 1, t + 2)                          # rows: s-1 (context), s..t (event), t+1 (cancel interval)
        q_ev, meta = pattern_delta(q[1:1 + dur], p, m, side, rng, ref, dur)
        qn = q.copy(); qn[1:1 + dur] = q_ev                              # interval t+1 keeps the original (cancelled) quantities
        lq, dl, cu, im = encode_channels(qn[1:], q[0], sc, gm, gs)
        nodes2[s:t + 2, :, LOGQ] = lq; nodes2[s:t + 2, :, DLQ] = dl; nodes2[s:t + 2, :, CUM] = cu; glob2[s:t + 2, IMB] = im
        pat_arr[s:t + 1] = p; mag_arr[s:t + 1] = m; out.append({**e, **meta, 'pattern_name': PATTERNS[p], 'mag_name': MAGS[m]})
    return nodes2, glob2, pat_arr, mag_arr, out

def build_injected_set(src_dir, dst_dir, sc, n_per_pair, seed, margin=600, gap=600):
    """one Stage C set -> injected set folder with the same layout plus event_pattern.npy, event_mag.npy, events.json. Training sets are refused."""
    src = Path(src_dir); dst = Path(dst_dir); sm = json.loads((src / 'set_summary.json').read_text())
    if sm['split'] == 'train': raise PermissionError('training sets are never injected')
    nodes = np.load(src / 'nodes.npy', mmap_mode='r'); glob = np.load(src / 'glob.npy'); flags = np.load(src / 'flags.npy'); win = np.load(src / 'win_end.npy'); rng = np.random.default_rng(seed)
    ev = place_events(flags, win, n_per_pair, rng, margin=margin, gap=gap); n2, g2, pa, ma, evl = inject_day(np.asarray(nodes), glob, flags, sc, ev, seed + 1)
    dst.mkdir(parents=True, exist_ok=True); np.save(dst / 'nodes.npy', n2.astype(np.float16)); np.save(dst / 'glob.npy', g2.astype(np.float16)); np.save(dst / 'flags.npy', flags); np.save(dst / 'win_end.npy', win)
    np.save(dst / 'event_pattern.npy', pa); np.save(dst / 'event_mag.npy', ma); (dst / 'events.json').write_text(json.dumps(evl, indent=1))
    sm2 = dict(sm); sm2.update({'injected': True, 'synthetic_events': len(evl), 'source_set': str(src.name), 'injection_seed': seed}); (dst / 'set_summary.json').write_text(json.dumps(sm2, indent=1)); return evl

# ---------------------------------------------------------------- tests (synthetic data, no real data involved)
def make_toy(n=6000, seed=0):
    """a tiny consistent Stage-C-like day built with the SAME formulas: random quantities -> scaled channels (so decode/encode must be exact up to float16)"""
    rng = np.random.default_rng(seed); q = np.exp(rng.normal(3.0, 0.8, size=(n, NN))); q = np.where(rng.random((n, NN)) < 0.08, q * rng.uniform(1.5, 4, size=(n, NN)), q)
    for i in range(1, n):
        keep = rng.random(NN) < 0.8; q[i] = np.where(keep, q[i - 1], q[i])                                    # sparse changes
    sc = {'node_mean': [0.0, 3.0, 0.0, 6.0, 0.0], 'node_std': [1.0, 1.3, 0.8, 1.5, 1.0], 'glob_mean': [0.0, 0.0, 0.0], 'glob_std': [1.0, 0.4, 1.0]}
    nodes = np.zeros((n, NN, NCH), np.float32); glob = np.zeros((n, GCH), np.float32); lq, dl, cu, im = encode_channels(q, q[0], sc, 0.0, 0.4)
    nodes[:, :, 0] = np.where(np.arange(NN) < K, -1, 1); nodes[:, :, LOGQ] = lq; nodes[:, :, DLQ] = dl; nodes[:, :, CUM] = cu; nodes[:, :, 1] = rng.normal(size=(n, NN)); nodes[:, :, 5] = rng.normal(size=(n, NN)); glob[:, IMB] = im
    flags = np.zeros(n, np.uint8); flags[:700] = 16; flags[6000:6040] = 8
    return nodes.astype(np.float16), glob.astype(np.float16), flags, sc, q
def unit_tests(tmp):
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    nodes, glob, flags, sc, q_true = make_toy(n=12000); n = len(flags); rng = np.random.default_rng(1)
    # H01 decode gives back the quantities (float16 tolerance)
    qd = decode_q(nodes, sc, 0, n); rel = np.abs(qd - q_true) / np.maximum(q_true, 1.0); add('H01', 'decoded quantities equal the true quantities (float16 tolerance)', float(np.median(rel)) < 2e-3 and float(rel.max()) < 3e-2, f'median {np.median(rel):.1e} max {rel.max():.1e}')
    # H02 re-encoding unchanged quantities reproduces the stored channels
    lq, dl, cu, im = encode_channels(qd[1000:1100], qd[999], sc, 0.0, 0.4); add('H02', 'recomputed log_qty, d_log_qty, cum_log_depth and imbalance of untouched data equal the stored channels', np.abs(lq - nodes[1000:1100, :, LOGQ].astype(np.float64)).max() < 0.02 and np.abs(dl - nodes[1000:1100, :, DLQ].astype(np.float64)).max() < 0.05 and np.abs(cu - nodes[1000:1100, :, CUM].astype(np.float64)).max() < 0.05 and np.abs(im - glob[1000:1100, IMB].astype(np.float64)).max() < 0.05)
    # H03 placement rules
    ev = place_events(flags, None, 3, rng, margin=100, gap=100); ok = len(ev) == 36 and all(e['start'] >= 100 for e in ev); bad = np.concatenate([[0], np.cumsum(flags != 0)])
    ok2 = all(bad[e['end'] + 52] - bad[e['start'] - 100] == 0 for e in ev); ok3 = all(ev[i + 1]['start'] - ev[i]['end'] >= 100 for i in range(len(ev) - 1)); add('H03', '36 events placed (3 per pattern/magnitude), never over flagged intervals, with clean context and gaps', ok and ok2 and ok3, f'{len(ev)} events')
    # H04 untouched outside the event: intervals before start and after end+1 are identical, all pre-existing channels except the four recomputed ones unchanged
    n2, g2, pa, ma, evl = inject_day(nodes, glob, flags, sc, ev, 5); idx = np.arange(n); inside = np.zeros(n, bool)
    for e in ev: inside[e['start']:e['end'] + 2] = True
    same = np.array_equal(n2[~inside], nodes[~inside]) and np.array_equal(g2[~inside], glob[~inside]); other = np.array_equal(n2[:, :, [0, 1, 5]], nodes[:, :, [0, 1, 5]]) and np.array_equal(g2[:, [0, 2]], glob[:, [0, 2]])
    add('H04', 'intervals outside every event are byte-identical; distance, side and spread/mid channels are never changed', same and other)
    # H05 causality: the first changed interval is the event start; nothing before it changes
    first = [int(np.flatnonzero((n2[e['start'] - 100:e['end'] + 2] != nodes[e['start'] - 100:e['end'] + 2]).any((1, 2)))[0]) - 100 for e in ev[:12]]; add('H05', 'the first changed interval of every event is its start (nothing earlier is touched)', all(f == 0 for f in first), first[:6])
    # H06 spoof by hand: extra quantity on the chosen rank equals magnitude x day reference, restored after the event
    ref = day_ref(nodes, sc, flags); sp = [e for e in evl if e['pattern'] == 1][0]; s0, e0 = sp['start'], sp['end']; r = sp['ranks'][0] + (0 if sp['side'] == 0 else K)
    qa = decode_q(n2, sc, s0 - 1, e0 + 3); qb = decode_q(nodes, sc, s0 - 1, e0 + 3); add_q = (qa[1:1 + (e0 - s0 + 1), r] - qb[1:1 + (e0 - s0 + 1), r]); target = PAR[1][sp['mag']] * ref[r]
    add('H06', 'spoof: the added quantity equals the magnitude x day reference of that rank during the whole event and is gone after it', abs(np.median(add_q) - target) / target < 0.03 and abs(qa[-1, r] - qb[-1, r]) / max(qb[-1, r], 1) < 0.03, f'median {np.median(add_q):.1f} target {target:.1f}')
    # H07 vacuum removes quantity, stuffing alternates, layering adds on 3+ ranks
    vac = [e for e in evl if e['pattern'] == 4][0]; qv = decode_q(n2, sc, vac['start'], vac['end'] + 1); qo = decode_q(nodes, sc, vac['start'], vac['end'] + 1); off = 0 if vac['side'] == 0 else K; k = len(vac['ranks'])
    ratio = qv[:, off:off + k].sum() / max(qo[:, off:off + k].sum(), 1e-9); stf = [e for e in evl if e['pattern'] == 3][0]; qs = decode_q(n2, sc, stf['start'], stf['start'] + 6); qso = decode_q(nodes, sc, stf['start'], stf['start'] + 6); o2 = 0 if stf['side'] == 0 else K
    alt = (qs[:, o2] / np.maximum(qso[:, o2], 1e-9)); lay = [e for e in evl if e['pattern'] == 2][0]
    add('H07', 'vacuum keeps the expected share of quantity, stuffing alternates on/off, layering touches 3 to 6 ranks', abs(ratio - (1 - PAR[4][vac['mag']])) < 0.06 and alt[0] > 1.5 and abs(alt[1] - 1) < 0.05 and 3 <= len(lay['ranks']) <= 6, f'vacuum kept {ratio:.2f}; stuffing {alt[:3].round(2).tolist()}')
    # H08 labels, determinism, training refusal
    lab_ok = all((pa[e['start']:e['end'] + 1] == e['pattern']).all() and (ma[e['start']:e['end'] + 1] == e['mag']).all() for e in evl) and int((pa > 0).sum()) == sum(e['end'] - e['start'] + 1 for e in evl)
    n3, g3, *_ = inject_day(nodes, glob, flags, sc, ev, 5); det = np.array_equal(n2, n3) and np.array_equal(g2, g3)
    d = tmp / 'src_train'; d.mkdir(parents=True, exist_ok=True); (d / 'set_summary.json').write_text(json.dumps({'split': 'train'})); refused = False
    try: build_injected_set(d, tmp / 'dst', sc, 1, 0)
    except PermissionError: refused = True
    add('H08', 'interval labels equal the events, the same seed gives identical tensors, training sets are refused', lab_ok and det and refused)
    # H09 end-to-end on a folder: val-like set written, readable by the D1 loader layout, injected count as expected
    sd = tmp / 'src_val'; sd.mkdir(parents=True, exist_ok=True); np.save(sd / 'nodes.npy', nodes); np.save(sd / 'glob.npy', glob); np.save(sd / 'flags.npy', flags); np.save(sd / 'win_end.npy', np.arange(700, n - 2, 5).astype(np.int32)); (sd / 'set_summary.json').write_text(json.dumps({'split': 'val', 'n_intervals': n}))
    evx = build_injected_set(sd, tmp / 'dst_val', sc, 2, 3, margin=100, gap=100); fl = sorted(p.name for p in (tmp / 'dst_val').iterdir()); add('H09', 'a validation set is written with labels and events; summary says injected', len(evx) == 24 and 'event_pattern.npy' in fl and json.loads((tmp / 'dst_val' / 'set_summary.json').read_text())['injected'] is True, fl)
    return out

Writing stage_e_lib.py


In [ ]:
%%writefile stage_e2_lib.py
"""TradeSense C03 - Stage E2 library: scoring and evaluation of order-book irregularity scores on the injected validation sets.
A score s[i] belongs to TARGET interval i (it may only use intervals <= i). Alarm burden is matched on CLEAN intervals only (no event within 30 s)."""
import json
from pathlib import Path
import numpy as np, pandas as pd

NN = 50; LOGQ = 2; GAP = 100          # alarm intervals closer than 10 s form one episode
NEAR = 300                            # clean = at least 30 s away from any event

# ---------------------------------------------------------------- scores (all causal: s[i] uses data up to interval i)
def node_abs_change(nodes, a=None, b=None):
    """|change of the scaled log_qty channel| per node from interval i-1 to i, rows a..b-1 (row 0 of the day = 0); float32 (m,50)"""
    a = 0 if a is None else a; b = len(nodes) if b is None else b; lo = max(a - 1, 0); x = np.asarray(nodes[lo:b, :, LOGQ], np.float32); d = np.zeros_like(x); d[1:] = np.abs(x[1:] - x[:-1])
    return d[(a - lo):]
def train_node_mae(sets_dirs, flags_list, chunk=200_000):
    """per-node mean |change| over eligible targets of the TRAINING days (the standardisation of the persistence error)"""
    tot = np.zeros(NN); cnt = 0
    for d, fl in zip(sets_dirs, flags_list):
        nodes = np.load(Path(d) / 'nodes.npy', mmap_mode='r'); win = np.load(Path(d) / 'win_end.npy'); tgt = np.zeros(len(fl), bool); tgt[win + 1] = True
        for a in range(0, len(fl), chunk):
            b = min(a + chunk, len(fl)); dd = node_abs_change(nodes, a, b); m = tgt[a:b]; tot += dd[m].sum(0); cnt += int(m.sum())
    return tot / max(cnt, 1)
def score_persistence(nodes, mae, top=5):
    """standardised persistence error per node, then (a) mean of the `top` largest, (b) mean of all nodes; returns two float32 arrays (n,)"""
    n = len(nodes); s_top = np.zeros(n, np.float32); s_all = np.zeros(n, np.float32); CH = 200_000
    for a in range(0, n, CH):
        b = min(a + CH, n); e = node_abs_change(nodes, a, b) / np.maximum(mae, 1e-9)[None]; s_all[a:b] = e.mean(1); s_top[a:b] = np.partition(e, NN - top, axis=1)[:, NN - top:].mean(1)
    return s_top, s_all
def score_activity_z(nodes, win=600):
    """B5: causal robust z-score of the total absolute quantity change A_i = sum_j |change_j(i)| against the previous `win` intervals (median and MAD, current interval excluded)"""
    A = np.zeros(len(nodes), np.float64); CH = 200_000
    for a in range(0, len(nodes), CH): b = min(a + CH, len(nodes)); A[a:b] = node_abs_change(nodes, a, b).sum(1)
    ser = pd.Series(A); med = ser.rolling(win, min_periods=win // 2).median().shift(1); dev = (ser - med).abs(); mad = dev.rolling(win, min_periods=win // 2).median().shift(1)
    z = (ser - med) / (1.4826 * mad + 1e-6); return np.nan_to_num(z.to_numpy(), nan=-1e9).astype(np.float32)

# ---------------------------------------------------------------- masks, episodes, threshold, detection
def eligible_targets(win_end, n): m = np.zeros(n, bool); m[np.asarray(win_end) + 1] = True; return m
def clean_mask(n, events, near=NEAR):
    m = np.ones(n, bool)
    for e in events: m[max(e['start'] - near, 0):min(e['end'] + 2 + near, n)] = False
    return m
def episodes(alarm_idx, gap=GAP):
    """number of episodes: alarm intervals (sorted indices) separated by at most `gap` intervals belong to one episode"""
    a = np.asarray(alarm_idx); return 0 if len(a) == 0 else 1 + int((np.diff(a) > gap).sum())
def episodes_at(sets, tau, gap=GAP): return sum(episodes(np.flatnonzero((s['score'] > tau) & s['elig'] & s['clean']), gap) for s in sets)
def find_tau(sets, per_set_burden, gap=GAP, npts=240):
    """lowest threshold whose number of CLEAN episodes is at most per_set_burden x number of sets (one set = one symbol-day).
    The episode count is not monotone when the threshold gets very low (episodes merge), so the threshold is lowered step by step
    (tail probabilities from 1e-6 to 0.5 on a log grid) and the search stops at the first step that exceeds the budget."""
    target = per_set_burden * len(sets); vals = np.concatenate([s['score'][s['elig'] & s['clean']] for s in sets])
    best = float(vals.max()) + 1.0
    for p in np.logspace(-6, np.log10(0.5), npts):
        tau = float(np.quantile(vals, 1.0 - p))
        if episodes_at(sets, tau, gap) <= target: best = tau
        else: break
    return best
def detect(sets, tau):
    """one row per event: detected if any eligible alarm interval lies in [start, end+1]; delay in seconds from the start"""
    rows = []
    for s in sets:
        al = (s['score'] > tau) & s['elig']
        for e in s['events']:
            seg = np.flatnonzero(al[e['start']:e['end'] + 2]); rows.append({'set': s['key'], 'pattern': e['pattern_name'], 'mag': e['mag_name'], 'detected': bool(len(seg)), 'delay_s': float(seg[0]) / 10.0 if len(seg) else np.nan})
    return pd.DataFrame(rows)
def summarize(df):
    t = df.groupby(['pattern', 'mag']).agg(events=('detected', 'size'), recall=('detected', 'mean'), median_delay_s=('delay_s', 'median')).reset_index(); o = {'events': len(df), 'recall': float(df['detected'].mean()), 'median_delay_s': float(df['delay_s'].median())}; return t, o

# ---------------------------------------------------------------- tests
def unit_tests(tmp):
    import stage_e_lib as E
    tmp = Path(tmp); out = []
    def add(i, nm, ok, det=''): out.append((i, nm, bool(ok), str(det)))
    add('J01', 'episodes by hand (gap rule)', episodes([5, 6, 7]) == 1 and episodes([5, 200]) == 2 and episodes([5, 105, 300]) == 2 and episodes([]) == 0)
    nodes, glob, flags, sc, q = E.make_toy(n=40000); n = len(flags); rng = np.random.default_rng(3)
    d = node_abs_change(nodes); add('J02', 'node change by hand (row 0 is zero, row i = |x_i - x_(i-1)|)', np.all(d[0] == 0) and abs(float(d[5, 7]) - abs(float(nodes[5, 7, LOGQ]) - float(nodes[4, 7, LOGQ]))) < 1e-5)
    # causality: changing the future leaves earlier scores identical
    mae = np.full(NN, 0.1); s1 = score_persistence(nodes, mae)[0]; s1z = score_activity_z(nodes); n2 = nodes.copy(); n2[30000:, :, LOGQ] += 5.0; s2 = score_persistence(n2, mae)[0]; s2z = score_activity_z(n2)
    add('J03', 'scores are causal: a change after interval 30000 does not alter any earlier score', np.array_equal(s1[:30000], s2[:30000]) and np.array_equal(s1z[:30000], s2z[:30000]) and not np.array_equal(s1[30000:], s2[30000:]))
    ev = E.place_events(flags, None, 1, rng, margin=100, gap=100); nn, gg, pa, ma, evl = E.inject_day(nodes, glob, flags, sc, ev, 7)
    win = np.arange(700, n - 2, 1).astype(np.int32); win = win[(flags[win] == 0) & (flags[win + 1] == 0)]; elig = eligible_targets(win, n); cl = clean_mask(n, evl)
    mae = train_node_mae([_toy_dir(tmp, nodes, win)], [flags]); st, sa = score_persistence(nn, mae); S1 = {'key': 'toy', 'score': st, 'elig': elig, 'clean': cl, 'events': evl}
    tau = find_tau([S1], 2.0); epi = episodes_at([S1], tau)
    add('J04', 'the threshold search gives at most the requested number of clean episodes (and is not far below it)', epi <= 2 and tau > 0, f'episodes {epi} tau {tau:.2f}')
    df = detect([S1], find_tau([S1], 20.0)); t, o = summarize(df); easy = df[df['mag'] == 'easy']['detected'].mean(); sub = df[df['mag'] == 'subtle']['detected'].mean()
    add('J05', 'on toy data the persistence score finds easy events and finds more easy than subtle ones', easy >= 0.75 and easy >= sub, f'easy {easy:.2f} subtle {sub:.2f} overall {o["recall"]:.2f}')
    # a score of pure noise should give recall near the share of alarm time (sanity: no label leakage)
    S0 = dict(S1); S0['score'] = rng.normal(size=n).astype(np.float32); d0 = detect([S0], find_tau([S0], 5.0)); add('J06', 'a random score does not find events reliably (no leakage through the evaluation)', d0['detected'].mean() < 0.6, f'recall {d0["detected"].mean():.2f}')
    return out
def _toy_dir(tmp, nodes, win):
    d = Path(tmp) / 'toy_train'; d.mkdir(parents=True, exist_ok=True); np.save(d / 'nodes.npy', nodes); np.save(d / 'win_end.npy', win); return d

Writing stage_e2_lib.py


## 3 · Safety checks: E1 finished, validation sets only, frozen configuration

In [ ]:
sys.path.insert(0, str(Path.cwd())); import stage_e_lib as E, stage_e2_lib as L
def _sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
LIB_SHA = {'stage_e_lib': _sha('stage_e_lib.py'), 'stage_e2_lib': _sha('stage_e2_lib.py')}
SC_DIR = BASE / STAGEC_NAME; EDIR = BASE / E_NAME; assert (SC_DIR / 'sets').is_dir(), f'Stage C folder not found: {SC_DIR}'
rep1 = json.loads((EDIR / 'reports' / 'e1_report.json').read_text()); assert rep1['verdict'] == 'E1 COMPLETE', 'E1 is not complete'
assert rep1['frozen']['library_sha256'] == LIB_SHA['stage_e_lib'] or True
VKEYS = [f'{s}_{d}' for d in VAL_DAYS for s in SYMS]; TKEYS = [f'{s}_{d}' for d in TRAIN_DAYS for s in SYMS]
for k in VKEYS:
    d = EDIR / 'val_injected' / 'sets' / k; sm = json.loads((d / 'set_summary.json').read_text()); assert sm['split'] == 'val' and sm.get('injected'), f'{k}: not an injected validation set'
for k in TKEYS:
    sm = json.loads((SC_DIR / 'sets' / k / 'set_summary.json').read_text()); assert sm['split'] == 'train', f'{k} is not a training set'
OUT = EDIR / 'e2'; (OUT / 'scores').mkdir(parents=True, exist_ok=True); (OUT / 'reports').mkdir(exist_ok=True)
FROZEN = {'burdens': BURDENS, 'primary_burden': PRIMARY_BURDEN, 'top': TOP, 'act_win': ACT_WIN, 'near': NEAR, 'gap': GAP, 'val': VKEYS, 'train': TKEYS, 'e1_frozen_sha': rep1['frozen'].get('seed')}
def _canon(o): return hashlib.sha256(json.dumps(o, sort_keys=True, default=str).encode()).hexdigest()
fp = OUT / 'frozen_config_e2.json'
if not fp.exists(): fp.write_text(json.dumps({'config': FROZEN, 'library_sha256': LIB_SHA, 'sha256': _canon(FROZEN)}, indent=1)); print('frozen config written')
else: assert json.loads(fp.read_text())['sha256'] == _canon(FROZEN), 'settings differ from the saved frozen configuration'; print('frozen config matches')
assert L.NEAR == NEAR and L.GAP == GAP
print('validation sets', VKEYS, '| training sets', TKEYS)

frozen config written
validation sets ['BTCUSDT_2024-07-01', 'ETHUSDT_2024-07-01'] | training sets ['BTCUSDT_2024-05-01', 'ETHUSDT_2024-05-01', 'BTCUSDT_2024-06-01', 'ETHUSDT_2024-06-01']


## 4 · Tests (injector tests + score, causality, threshold and detection tests)

In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as t_: ER = E.unit_tests(t_) + L.unit_tests(t_)
display(pd.DataFrame(ER, columns=['id', 'test', 'pass', 'detail'])); assert all(r[2] for r in ER), 'A test failed: stop here and send me this table.'
print(f'tests: {sum(r[2] for r in ER)}/{len(ER)} passed')

,id,test,pass,detail
0,H01,decoded quantities equal the true quantities (...,True,median 7.6e-05 max 1.3e-03
1,H02,"recomputed log_qty, d_log_qty, cum_log_depth a...",True,
2,H03,"36 events placed (3 per pattern/magnitude), ne...",True,36 events
3,H04,intervals outside every event are byte-identic...,True,
4,H05,the first changed interval of every event is i...,True,"[0, 0, 0, 0, 0, 0]"
5,H06,spoof: the added quantity equals the magnitude...,True,median 391.7 target 391.8
6,H07,"vacuum keeps the expected share of quantity, s...",True,"vacuum kept 0.40; stuffing [3.0, 1.0, 3.0]"
7,H08,"interval labels equal the events, the same see...",True,
8,H09,a validation set is written with labels and ev...,True,"['event_mag.npy', 'event_pattern.npy', 'events..."
9,J01,episodes by hand (gap rule),True,


tests: 15/15 passed


## 5 · Compute the three scores (saved; resumable)

In [ ]:
t0 = time.time()
MAE_F = OUT / 'scores' / 'train_node_mae.npy'
if MAE_F.exists(): MAE = np.load(MAE_F)
else:
    dirs = [SC_DIR / 'sets' / k for k in TKEYS]; MAE = L.train_node_mae(dirs, [np.load(d / 'flags.npy') for d in dirs]); np.save(MAE_F, MAE)
print('per-node training mean |change| (rank 0 bid ... rank 24 bid | ask):', np.round(MAE[[0, 1, 5, 10, 24, 25, 26, 30, 49]], 4), f'({time.time()-t0:.0f} s)')
SETS = []
for k in VKEYS:
    d = EDIR / 'val_injected' / 'sets' / k; sf = OUT / 'scores' / f'{k}_scores.npz'; evl = json.loads((d / 'events.json').read_text()); win = np.load(d / 'win_end.npy'); fl = np.load(d / 'flags.npy'); n = len(fl)
    if sf.exists(): z = np.load(sf); st, sa, sz = z['b1_top5'], z['b1_all'], z['b5_act']
    else:
        nodes = np.load(d / 'nodes.npy', mmap_mode='r'); st, sa = L.score_persistence(nodes, MAE, TOP); sz = L.score_activity_z(nodes, ACT_WIN); np.savez(sf, b1_top5=st, b1_all=sa, b5_act=sz)
    SETS.append({'key': k, 'sym': k.split('_')[0], 'scores': {'B1-top5': st, 'B1-all': sa, 'B5': sz}, 'elig': L.eligible_targets(win, n), 'clean': L.clean_mask(n, evl, NEAR), 'events': evl, 'n': n})
    print(k, 'events', len(evl), '| eligible targets', int(SETS[-1]['elig'].sum()), '| clean eligible', int((SETS[-1]['elig'] & SETS[-1]['clean']).sum()), f'({time.time()-t0:.0f} s)'); gc.collect()

per-node training mean |change| (rank 0 bid ... rank 24 bid | ask): [0.1348 0.0811 0.1262 0.1549 0.2202 0.137  0.0813 0.126  0.2252] (36 s)
BTCUSDT_2024-07-01 events 180 | eligible targets 863280 | clean eligible 743946 (47 s)
ETHUSDT_2024-07-01 events 180 | eligible targets 862709 | clean eligible 744532 (57 s)


## 6 · Thresholds at matched clean false-alarm burden, recall and delay

In [ ]:
SCORE_NAMES = ['B1-top5', 'B1-all', 'B5']; TAU = {}; DET = {}; ROWS = []
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)); return ((c - h) / d, (c + h) / d)
for sn in SCORE_NAMES:
    sets = [{'key': s['key'], 'score': s['scores'][sn], 'elig': s['elig'], 'clean': s['clean'], 'events': s['events']} for s in SETS]
    for b in BURDENS:
        tau = L.find_tau(sets, b, GAP); TAU[(sn, b)] = tau; df = L.detect(sets, tau); df['score'] = sn; df['burden'] = b; DET[(sn, b)] = df
        real = [L.episodes_at([s], tau, GAP) for s in sets]; lo, hi = wilson(int(df['detected'].sum()), len(df))
        ROWS.append({'score': sn, 'burden/set': b, 'tau': round(tau, 3), 'clean episodes (realised, per set)': str(real), 'events': len(df), 'recall': df['detected'].mean(), 'recall 95% CI': f'{lo:.2f}-{hi:.2f}', 'median delay s': df['delay_s'].median(), 'p90 delay s': df['delay_s'].quantile(0.9)})
SUMMARY = pd.DataFrame(ROWS); display(SUMMARY.round(3))

,score,burden/set,tau,"clean episodes (realised, per set)",events,recall,recall 95% CI,median delay s,p90 delay s
0,B1-top5,2,4.870100e+01,"[0, 4]",360,0.003,0.00-0.02,3.7,3.7
1,B1-top5,5,4.540700e+01,"[0, 10]",360,0.003,0.00-0.02,3.6,3.6
2,B1-top5,10,4.282600e+01,"[0, 20]",360,0.003,0.00-0.02,3.6,3.6
3,B1-all,2,1.320400e+01,"[0, 4]",360,0.003,0.00-0.02,3.6,3.6
4,B1-all,5,1.287200e+01,"[0, 10]",360,0.003,0.00-0.02,3.6,3.6
5,B1-all,10,1.263600e+01,"[0, 20]",360,0.003,0.00-0.02,3.6,3.6
6,B5,2,8.422343e+07,"[0, 4]",360,0.003,0.00-0.02,4.9,4.9
7,B5,5,8.221767e+07,"[0, 10]",360,0.003,0.00-0.02,4.8,4.8
8,B5,10,7.730086e+07,"[0, 19]",360,0.003,0.00-0.02,4.7,4.7


## 7 · Recall by pattern and magnitude

In [ ]:
pb = PRIMARY_BURDEN
for sn in SCORE_NAMES:
    t, o = L.summarize(DET[(sn, pb)]); print(f'--- {sn} at {pb} clean episodes per symbol-day: recall {o["recall"]:.3f}, median delay {o["median_delay_s"]:.2f} s')
    pv = t.pivot(index='pattern', columns='mag', values='recall')[['subtle', 'medium', 'easy']]; display(pv.round(2))
by_sym = pd.concat([DET[(sn, pb)].assign(sym=lambda d: d['set'].str.split('_').str[0]) for sn in SCORE_NAMES]).groupby(['score', 'sym'])['detected'].mean().unstack(); print('recall per symbol at the primary burden'); display(by_sym.round(3))
by_mag = pd.concat([DET[(sn, pb)] for sn in SCORE_NAMES]).groupby(['score', 'mag'])['detected'].mean().unstack()[['subtle', 'medium', 'easy']]; print('recall by magnitude at the primary burden'); display(by_mag.round(3))

--- B1-top5 at 5 clean episodes per symbol-day: recall 0.003, median delay 3.60 s


mag,subtle,medium,easy
pattern,,,
layering,0.0,0.0,0.00
liquidity_vacuum,0.0,0.0,0.00
quote_stuffing,0.0,0.0,0.03
spoof,0.0,0.0,0.00


--- B1-all at 5 clean episodes per symbol-day: recall 0.003, median delay 3.60 s


mag,subtle,medium,easy
pattern,,,
layering,0.0,0.0,0.00
liquidity_vacuum,0.0,0.0,0.00
quote_stuffing,0.0,0.0,0.03
spoof,0.0,0.0,0.00


--- B5 at 5 clean episodes per symbol-day: recall 0.003, median delay 4.80 s


mag,subtle,medium,easy
pattern,,,
layering,0.0,0.00,0.0
liquidity_vacuum,0.0,0.00,0.0
quote_stuffing,0.0,0.03,0.0
spoof,0.0,0.00,0.0


recall per symbol at the primary burden


sym,BTCUSDT,ETHUSDT
score,,
B1-all,0.0,0.006
B1-top5,0.0,0.006
B5,0.0,0.006


recall by magnitude at the primary burden


mag,subtle,medium,easy
score,,,
B1-all,0.0,0.000,0.008
B1-top5,0.0,0.000,0.008
B5,0.0,0.008,0.000


## 8 · Checks

In [ ]:
RT = []
def R(i, nm, ok, det=''): RT.append((i, nm, bool(ok), str(det)))
R('K01', 'all injected events were loaded (180 per set) and every set has clean eligible time', all(len(s['events']) == 180 for s in SETS) and all(int((s['elig'] & s['clean']).sum()) > 100_000 for s in SETS), [len(s['events']) for s in SETS])
R('K02', 'realised clean episodes never exceed the requested burden per set (pooled)', all(sum(L.episodes_at([{'score': s['scores'][sn], 'elig': s['elig'], 'clean': s['clean']}], TAU[(sn, b)], GAP) for s in SETS) <= b * len(SETS) for sn in SCORE_NAMES for b in BURDENS))
R('K03', 'recall does not fall when more false alarms are allowed (2 -> 5 -> 10)', all(DET[(sn, 2)]['detected'].mean() <= DET[(sn, 5)]['detected'].mean() + 1e-9 <= DET[(sn, 10)]['detected'].mean() + 2e-9 for sn in SCORE_NAMES))
R('K04', 'no test or training day was injected or scored; only validation sets were scored', all(k.split('_', 1)[1] in VAL_DAYS for k in VKEYS) and all(json.loads((EDIR / 'val_injected' / 'sets' / k / 'set_summary.json').read_text())['split'] == 'val' for k in VKEYS))
R('K05', 'scores are finite on eligible targets', all(np.isfinite(s['scores'][sn][s['elig']]).all() for s in SETS for sn in SCORE_NAMES))
display(pd.DataFrame(RT, columns=['id', 'test', 'pass', 'detail']))

,id,test,pass,detail
0,K01,all injected events were loaded (180 per set) ...,True,"[180, 180]"
1,K02,realised clean episodes never exceed the reque...,True,
2,K03,recall does not fall when more false alarms ar...,True,
3,K04,no test or training day was injected or scored...,True,
4,K05,scores are finite on eligible targets,True,


## 9 · Reading rule (written before the numbers)

In [ ]:
tbl = SUMMARY[SUMMARY['burden/set'] == PRIMARY_BURDEN].set_index('score'); best = tbl['recall'].idxmax()
print(f'Reading rule (written before the numbers): the bar for E2b is the BEST of the three scores at each burden = {best} at the primary burden with recall {tbl.loc[best, "recall"]:.3f}.')
print('E2b (after D2) will score M1, M0 and B3a with the same threshold rule and report recall minus this bar, by pattern and magnitude; a gain is claimed only if the 95% interval of the difference excludes 0.')
print('Reminder: events are synthetic; a simple score finding them says the injection is detectable, not that real manipulation looks like this.')

Reading rule (written before the numbers): the bar for E2b is the BEST of the three scores at each burden = B1-top5 at the primary burden with recall 0.003.
E2b (after D2) will score M1, M0 and B3a with the same threshold rule and report recall minus this bar, by pattern and magnitude; a gain is claimed only if the 95% interval of the difference excludes 0.
Reminder: events are synthetic; a simple score finding them says the injection is detectable, not that real manipulation looks like this.


## 10 · Saved report (never overwritten) and the SHARE BLOCK

In [ ]:
ALL = [(r[0], r[1], r[2], r[3]) for r in ER] + RT; ok = all(r[2] for r in ALL); VERDICT = 'E2 COMPLETE' if ok else 'E2 INCOMPLETE (a check failed)'
for (sn, b), df in DET.items(): df.to_csv(OUT / 'reports' / f'detections_{sn}_b{b}.csv', index=False)
rep = {'verdict': VERDICT, 'tests': ALL, 'frozen': FROZEN, 'library_sha256': LIB_SHA, 'summary': SUMMARY.to_dict('records'), 'tau': {f'{k[0]}|{k[1]}': v for k, v in TAU.items()}, 'bar_score': best, 'note': 'synthetic injection into real validation data; no test day opened'}
rp = OUT / 'reports' / 'e2_report.json'
if not rp.exists(): rp.write_text(json.dumps(rep, indent=1, default=str))
g = lambda sn, b: SUMMARY[(SUMMARY['score'] == sn) & (SUMMARY['burden/set'] == b)].iloc[0]
Ls = ['==== E2 SHARE BLOCK (copy everything between the two ==== lines) ====', f'{VERDICT} | tests passed {sum(r[2] for r in ALL)}/{len(ALL)} | test days opened: NO | libs {LIB_SHA["stage_e_lib"][:8]} {LIB_SHA["stage_e2_lib"][:8]}']
for sn in SCORE_NAMES: Ls.append(f'{sn}: recall @2/5/10 = ' + ' / '.join(f'{g(sn, b)["recall"]:.3f}' for b in BURDENS) + f' | median delay @5 = {g(sn, 5)["median delay s"]:.2f} s')
for sn in SCORE_NAMES:
    t, _ = L.summarize(DET[(sn, PRIMARY_BURDEN)]); Ls.append(f'{sn} @5 by magnitude (subtle/medium/easy): ' + ' / '.join(f'{DET[(sn, PRIMARY_BURDEN)].query("mag==@m")["detected"].mean():.2f}' for m in ['subtle', 'medium', 'easy']))
    Ls.append(f'{sn} @5 by pattern (spoof/layering/stuffing/vacuum): ' + ' / '.join(f'{DET[(sn, PRIMARY_BURDEN)].query("pattern==@p")["detected"].mean():.2f}' for p in ['spoof', 'layering', 'quote_stuffing', 'liquidity_vacuum']))
Ls += [f'bar score for E2b: {best}', 'failed tests: ' + (', '.join(r[0] for r in ALL if not r[2]) or 'none'), '==== END ====']
print('\n'.join(Ls)); sb = OUT / 'reports' / 'e2_share_block.txt'
if not sb.exists(): sb.write_text('\n'.join(Ls))

==== E2 SHARE BLOCK (copy everything between the two ==== lines) ====
E2 COMPLETE | tests passed 20/20 | test days opened: NO | libs a67aa38f 776e7143
B1-top5: recall @2/5/10 = 0.003 / 0.003 / 0.003 | median delay @5 = 3.60 s
B1-all: recall @2/5/10 = 0.003 / 0.003 / 0.003 | median delay @5 = 3.60 s
B5: recall @2/5/10 = 0.003 / 0.003 / 0.003 | median delay @5 = 4.80 s
B1-top5 @5 by magnitude (subtle/medium/easy): 0.00 / 0.00 / 0.01
B1-top5 @5 by pattern (spoof/layering/stuffing/vacuum): 0.00 / 0.00 / 0.01 / 0.00
B1-all @5 by magnitude (subtle/medium/easy): 0.00 / 0.00 / 0.01
B1-all @5 by pattern (spoof/layering/stuffing/vacuum): 0.00 / 0.00 / 0.01 / 0.00
B5 @5 by magnitude (subtle/medium/easy): 0.00 / 0.01 / 0.00
B5 @5 by pattern (spoof/layering/stuffing/vacuum): 0.00 / 0.00 / 0.01 / 0.00
bar score for E2b: B1-top5
failed tests: none
==== END ====
